In [5]:
import sys
sys.executable


'C:\\Users\\bktpk\\.conda\\envs\\analytics\\python.exe'

In [6]:

import pandas as pd
import numpy as np

#loading data
events = pd.read_csv('events.csv')
experiments = pd.read_csv('experiments.csv')
subscriptions = pd.read_csv('subscriptions.csv')
users = pd.read_csv('users.csv')
print(f" events dataset: {events.shape[0]} rows, {events.shape[1]} columns")
print(f" experiment dataset:{experiments.shape[0]} rows, {experiments.shape[1]} columns")
print(f" subscriptions dataset:{subscriptions.shape[0]} rows, {subscriptions.shape[1]} columns")
print(f" users dataset: {users.shape[0]} rows, {users.shape[1]} columns")

 events dataset: 550000 rows, 4 columns
 experiment dataset:7500 rows, 5 columns
 subscriptions dataset:20000 rows, 6 columns
 users dataset: 50000 rows, 5 columns


In [7]:
# cleaning header
print(f" events columns: {events.columns.tolist()}")
print(f" experiments columns: {experiments.columns.tolist()}")
print(f" subscription columns: {subscriptions.columns.tolist()}")
print(f" users columns: {users.columns.tolist()}")

 events columns: ['event_id', 'user_id', 'event_type', 'event_timestamp']
 experiments columns: ['experiment_id', 'user_id', 'variant', 'metric_value', 'experiment_date']
 subscription columns: ['subscription_id', 'user_id', 'plan_type', 'start_date', 'monthly_price', 'end_date']
 users columns: ['user_id', 'signup_date', 'acquisition_channel', 'country', 'device_type']


In [8]:
# check for categorical typos
print(f" events_event_type {events['event_type'].unique()}")
print(f" experiments_variant {experiments['variant'].unique()}")
print(f" subscription_plan_type {subscriptions['plan_type'].unique()}")
print(f" users_acquisition {users['acquisition_channel'].unique()}")
print(f" userss_country {users['country'].unique()}")
print(f" users_device_type {users['device_type'].unique()}")

 events_event_type <StringArray>
['login', 'content_view', 'action', 'signup']
Length: 4, dtype: str
 experiments_variant <StringArray>
['control', 'treatment']
Length: 2, dtype: str
 subscription_plan_type <StringArray>
['basic', 'pro']
Length: 2, dtype: str
 users_acquisition <StringArray>
['paid', 'organic', 'referral', 'social']
Length: 4, dtype: str
 userss_country <StringArray>
['CA', 'UK', 'AU', 'IN', 'US']
Length: 5, dtype: str
 users_device_type <StringArray>
['ios', 'web', 'android']
Length: 3, dtype: str


In [9]:
events.isnull().sum()


event_id           0
user_id            0
event_type         0
event_timestamp    0
dtype: int64

In [10]:
experiments.isnull().sum()


experiment_id      0
user_id            0
variant            0
metric_value       0
experiment_date    0
dtype: int64

In [11]:
subscriptions.isnull().sum()

subscription_id       0
user_id               0
plan_type             0
start_date            0
monthly_price         0
end_date           3276
dtype: int64

In [14]:
#checking date format
print(events['event_timestamp'].dtype)
events['event_timestamp'] = pd.to_datetime(events['event_timestamp'], errors = 'coerce')
print("fix applied")

datetime64[us]
fix applied


In [15]:
#date format experiments
print(experiments['experiment_date'].dtype)
experiments['experiment_date'] = pd.to_datetime(experiments['experiment_date'], errors = 'coerce')
print("fix applied")

datetime64[us]
fix applied


In [17]:
#date format subscriptions
print(subscriptions['start_date'].dtype)
print(subscriptions['end_date'].dtype)
subscriptions['start_date'] = pd.to_datetime(subscriptions['start_date'], errors= 'coerce')
print("start date fix applied")
subscriptions['end_date'] = pd.to_datetime(subscriptions['end_date'], errors= 'coerce')
print("end date fix applied")

datetime64[us]
datetime64[us]
start date fix applied
end date fix applied


In [19]:
# date format users
print(users['signup_date'].dtype)
users['signup_date'] = pd.to_datetime(users['signup_date'], errors = 'coerce')
print("fix applied")

datetime64[us]
fix applied


In [20]:
#seperate event_timestamp into date and time columns
events['event_date'] = events['event_timestamp'].dt.date
print("date added")
events['event_time'] = events['event_timestamp'].dt.time
print("time added")
events['event_hour'] = events['event_timestamp'].dt.hour
print("hour added")


date added
time added
hour added


In [21]:
print(subscriptions)

       subscription_id  user_id plan_type start_date  monthly_price   end_date
0                    1    23971     basic 2024-11-23           10.0 2025-01-22
1                    2    47815     basic 2024-08-28           10.0 2024-10-27
2                    3    35759       pro 2024-06-28           25.0 2024-09-26
3                    4    14702     basic 2024-04-14           10.0 2024-10-11
4                    5    46787     basic 2024-02-11           10.0 2024-08-09
...                ...      ...       ...        ...            ...        ...
19995            19996    22196     basic 2024-05-16           10.0 2024-06-15
19996            19997    14282       pro 2024-06-14           25.0        NaT
19997            19998    41619     basic 2023-12-09           10.0        NaT
19998            19999    11686       pro 2023-07-31           25.0 2023-08-30
19999            20000    31563     basic 2023-04-09           10.0        NaT

[20000 rows x 6 columns]


In [22]:
# subscription time
time_logic = subscriptions['start_date'] < subscriptions['end_date']
print(subscriptions.loc[time_logic,['subscription_id', 'start_date', 'end_date']].head(3))

   subscription_id start_date   end_date
0                1 2024-11-23 2025-01-22
1                2 2024-08-28 2024-10-27
2                3 2024-06-28 2024-09-26


In [23]:
events.head(10)

,event_id,user_id,event_type,event_timestamp,event_date,event_time,event_hour
0,1,13947,login,2024-06-09 14:36:00,2024-06-09,14:36:00,14.0
1,2,5227,content_view,2024-08-08 20:15:00,2024-08-08,20:15:00,20.0
2,3,45251,login,2024-12-17 07:45:00,2024-12-17,07:45:00,7.0
3,4,42855,action,2023-12-24 18:27:00,2023-12-24,18:27:00,18.0
4,5,3397,login,2024-10-14 15:38:00,2024-10-14,15:38:00,15.0
5,6,49162,content_view,2023-01-21 06:04:00,2023-01-21,06:04:00,6.0
6,7,31068,content_view,2024-08-31 00:25:00,2024-08-31,00:25:00,0.0
7,8,12439,content_view,2024-12-15 12:13:00,2024-12-15,12:13:00,12.0
8,9,30274,login,2024-02-03 19:53:00,2024-02-03,19:53:00,19.0
9,10,40188,content_view,2024-10-29 19:19:00,2024-10-29,19:19:00,19.0


In [24]:
from sqlalchemy import create_engine
import os
from datetime import datetime
import logging
from config import DB_CONFIG

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Database connection
engine = create_engine(
    f"postgresql+psycopg2://{DB_CONFIG['username']}:{DB_CONFIG['password']}@"
    f"{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

logging.info("Database connection created")



2026-04-26 21:23:32,697 - INFO - Database connection created


In [25]:

# Load data to PostgreSQL

print("\nLoading transformed data to PostgreSQL")

events.to_sql('events', engine, if_exists='replace', index=False)
logging.info(f"events -> {len(events)} rows")

experiments.to_sql('experiments', engine, if_exists='replace', index=False)
logging.info(f"experiments -> {len(experiments)} rows")

subscriptions.to_sql('subscriptions', engine, if_exists='replace', index=False)
logging.info(f"subscriptions -> {len(subscriptions)} rows")

users.to_sql('users', engine, if_exists='replace', index=False)
logging.info(f"users -> {len(users)} rows")

print("Data loaded")


Loading transformed data to PostgreSQL


2026-04-26 21:23:59,004 - INFO - events -> 550000 rows
2026-04-26 21:23:59,251 - INFO - experiments -> 7500 rows
2026-04-26 21:23:59,978 - INFO - subscriptions -> 20000 rows
2026-04-26 21:24:01,264 - INFO - users -> 50000 rows


Data loaded successfully!
